In [ ]:
# 目标: 生成 p >> n 的 X_train_ultrahigh 和 X_test_ultrahigh
library(splines)

train_data <- readRDS("../data/01_Train_Data.rds")
test_data  <- readRDS("../data/01_Test_Data.rds")

X_train <- train_data$X
X_test  <- test_data$X

# --- 经济学特征分块索引 (1到94) ---
idx_profit     <- c(1:12, 15, 42, 43, 68, 69, 86, 89, 90)
idx_leverage   <- c(14, 35:41, 60, 64, 66, 67, 77:79, 84, 85, 91:93, 94)
idx_cashflow   <- c(13, 20, 32:34, 54:59, 61:63, 65, 74, 75, 80:83)
idx_efficiency <- c(44:53, 70:73, 76, 87, 88)
idx_growth     <- c(16:19, 21:31)

In [ ]:

expand_splines_strict <- function(train_mat, test_mat, df_spline = 3) {
    
    train_list <- list()
    test_list  <- list()
    
    for (i in 1:ncol(train_mat)) {
        var_name <- colnames(train_mat)[i]
        x_train <- train_mat[, i]
        x_test  <- test_mat[, i]
        
        # 护盾：使用 tryCatch 捕捉样条展开时的节点重合错误
        basis_train <- tryCatch({
            ns(x_train, df = df_spline)
        }, error = function(e) {
            # 如果报错，不中断程序，而是返回 NULL 交给下游处理
            return(NULL)
        })
        
        # 判断：如果顺利生成了样条
        if (!is.null(basis_train)) {
            colnames(basis_train) <- paste0(var_name, "_ns", 1:df_spline)
            
            # 测试集同步映射
            basis_test <- predict(basis_train, x_test)
            colnames(basis_test) <- paste0(var_name, "_ns", 1:df_spline)
            
        } else {
            # 降级处理：说明数据过于稀疏或倾斜，放弃样条展开，退化为一次项(原始变量)
            cat(sprintf("   [降级提示] 变量 %s 数据极端分布(节点重合)，退化为线性项。\n", var_name))
            
            basis_train <- matrix(x_train, ncol = 1)
            colnames(basis_train) <- paste0(var_name, "_linear")
            
            basis_test <- matrix(x_test, ncol = 1)
            colnames(basis_test) <- paste0(var_name, "_linear")
        }
        
        train_list[[i]] <- basis_train
        test_list[[i]]  <- basis_test
    }
    
    return(list(Train = do.call(cbind, train_list), 
                Test  = do.call(cbind, test_list)))
}

spline_profit     <- expand_splines_strict(X_train[, idx_profit], X_test[, idx_profit])
spline_leverage   <- expand_splines_strict(X_train[, idx_leverage], X_test[, idx_leverage])
spline_cashflow   <- expand_splines_strict(X_train[, idx_cashflow], X_test[, idx_cashflow])
spline_efficiency <- expand_splines_strict(X_train[, idx_efficiency], X_test[, idx_efficiency])
spline_growth     <- expand_splines_strict(X_train[, idx_growth], X_test[, idx_growth])


In [ ]:
tensor_product <- function(mat1, mat2, prefix="TP") {
  n <- nrow(mat1)
  p1 <- ncol(mat1)
  p2 <- ncol(mat2)
  
  # 初始化结果矩阵和列名向量
  result <- matrix(0, nrow = n, ncol = p1 * p2)
  col_names <- character(p1 * p2)
  
  col_idx <- 1
  for (i in 1:p1) {
    for (j in 1:p2) {
      # 核心逻辑：两两列相乘
      result[, col_idx] <- mat1[, i] * mat2[, j]
      # 自动生成具有经济学解释性的变量名
      col_names[col_idx] <- paste0(prefix, "_", colnames(mat1)[i], "_X_", colnames(mat2)[j])
      col_idx <- col_idx + 1
    }
  }
  
  colnames(result) <- col_names
  return(result)
}

In [ ]:
# 将训练集和测试集的样条结果放入 List，方便循环调用
list_spline_train <- list(
  Prof = spline_profit$Train,
  Lev  = spline_leverage$Train,
  Cash = spline_cashflow$Train,
  Eff  = spline_efficiency$Train,
  Grow = spline_growth$Train
)

list_spline_test <- list(
  Prof = spline_profit$Test,
  Lev  = spline_leverage$Test,
  Cash = spline_cashflow$Test,
  Eff  = spline_efficiency$Test,
  Grow = spline_growth$Test
)

block_names <- names(list_spline_train)
num_blocks <- length(block_names)

# 初始化空列表，用于存放生成的张量积矩阵
TP_train_list <- list()
TP_test_list  <- list()

# 双重循环，自动计算 C(5, 2) = 10 组两两交互
pair_idx <- 1
for (i in 1:(num_blocks - 1)) {
  for (j in (i + 1):num_blocks) {
    name1 <- block_names[i]
    name2 <- block_names[j]
    prefix <- paste0(name1, "_", name2)
    
    cat(sprintf("正在计算第 %d/10 组: %s × %s ...\n", pair_idx, name1, name2))
    
    # 训练集交叉
    TP_train_list[[prefix]] <- tensor_product(list_spline_train[[i]], list_spline_train[[j]], prefix)
    # 测试集交叉
    TP_test_list[[prefix]]  <- tensor_product(list_spline_test[[i]],  list_spline_test[[j]],  prefix)
    
    pair_idx <- pair_idx + 1
  }
}

# 1. 拼接训练集: 原始 X + 单变量样条 + 10组张量积
X_train_ultrahigh <- do.call(cbind, c(
  list(X_train), 
  list_spline_train, 
  TP_train_list
))

# 2. 拼接测试集: 原始 X + 单变量样条 + 10组张量积
X_test_ultrahigh <- do.call(cbind, c(
  list(X_test), 
  list_spline_test, 
  TP_test_list
))

# 将组装好的矩阵连同标签一起保存，方便 SIS 脚本直接读取
saveRDS(list(X = X_train_ultrahigh, Y = train_data$Y), "../data/02_Train_Ultrahigh.rds")
saveRDS(list(X = X_test_ultrahigh,  Y = test_data$Y),  "../data/02_Test_Ultrahigh.rds")

cat("训练集维度:", dim(X_train_ultrahigh), "\n")
cat("测试集维度:", dim(X_test_ultrahigh), "\n")

# 释放内存
rm(TP_train_list, TP_test_list, list_spline_train, list_spline_test)
gc()